# Getting started with beamforming

In this Notebook, we propose to perform a beamforming operation on signals from a NP array using the `Beamformer` class. 

## Beamforming from NP array signals


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from megamicros.log import log
from megamicros.core.np import MemsArray, MemsArrayNP
from megamicros.tools.acoustics.bmf import BeamformerFDAS
from megamicros.tools.geometry import circle, horizontalPlan
from megamicros.data import generate_movie

DIRECTORY = "./"

log.setLevel( "INFO" )

Let's create an array of random data aiming at simulate signals from a 32 MEMs antenna.

In [ ]:
data = np.random.rand( 100000, 32 )
data.shape

Create the antenna

In [ ]:
antenna = MemsArrayNP( data )

Specify avalable the 32 MEMs channels and activate them:

In [ ]:
antenna.setAvailableMems( 32 )
antenna.setActiveMems( [i for i in range(32)] )

Run on two seconds duration.

In [ ]:
antenna.run( duration=2, datatype='float32' )

# get data, loop is blocking
for data in antenna:
    print( f"data={data}" )

# Wait until the thread terminates
antenna.wait()

### Init the beamformer

In [ ]:
# Define antenna geometry
mems_number = 32
antenna_radius = 0.25
antenna_thickness = 0.0
antenna_angle_offset = 2 * np.pi / mems_number / 2

mems_position = np.array( circle( points_number=mems_number, radius=antenna_radius, height=antenna_thickness, angle_offset=antenna_angle_offset, clockwise=True ) )

# Define antenna focal
focal_plan_width = 5.72
focal_plan_depth = 7.6
focal_depth = 1.9
focal_plan_width_sampling = 21
focal_plan_depth_sampling = 21

locations = np.array( horizontalPlan( focal_plan_width, focal_plan_depth, -focal_depth, focal_plan_width_sampling, focal_plan_depth_sampling ) )

# Create the beamformer
bmf = BeamformerFDAS( 
    mems_position = mems_position,
    locations = locations,
    sampling_frequency = antenna.sampling_frequency,
    frame_length = antenna.frame_length,
)


### Plot antenna form

In [ ]:
fig = plt.figure()
ax = fig.add_subplot( 131, projection='3d' )
ax.scatter( mems_position[:,0], mems_position[:,1], mems_position[:,2] )
ax.scatter( locations[:,0], locations[:,1], locations[:,2] )
ax = fig.add_subplot( 132, projection='3d' )

ax.set_xlabel( 'X' )
ax.set_ylabel( 'Y' )
ax.scatter( mems_position[:,0], mems_position[:,1], mems_position[:,2], c=np.arange(32) )

### Process beamforming

Beamforming result is stored in the BFE array

In [ ]:
antenna.run(
    mems = antenna.available_mems,
    duration=2,
    counter_skip = True,
    frame_length=512,
    datatype='int32',
)

sound = []
imgs = []
for data in antenna:
    sound.append( data[:, 1:3] )

    BFE, _, _ = bmf.compute( data )
    imgs.append( np.reshape(BFE, (focal_plan_depth_sampling, focal_plan_width_sampling) ) )

sound = np.concatenate( sound, axis=0 )
sound = sound.astype( np.float32 )*antenna.sensibility

### Make a BFE video

In [ ]:
generate_movie( 
    imgs, 
    rate=antenna.sampling_frequency/antenna.frame_length,
    sound=sound,
    sampling_frequency=antenna.sampling_frequency,
    norm=None,
    directory=DIRECTORY,
    colored=True
)

# display.Video( DIRECTORY + 'tmp/movie.mp4' )